In [2]:
from pathlib import Path
Path.cwd()

PosixPath('/Users/banti/Documents/Work/Agentic AI/research_assistant/app')

In [5]:
from functools import lru_cache
from pathlib import Path

from core.schemas import Subtask

from core.schemas import ResearchPlan
from models.planner_llm import get_planner_llm


PROMPT_PATH = Path.cwd() / "prompts" / "planner.md"


@lru_cache(maxsize=128)
def _load_prompt(path: str) -> str:
    return Path(path).read_text(encoding="utf-8")


class PlannerAgent:
    def __init__(self):
        self.llm = get_planner_llm()
        self.prompt_template = _load_prompt(str(PROMPT_PATH))

    def create_plan(self, query: str) -> ResearchPlan:
        prompt = self.prompt_template.format(query=query)
        structured_llm = self.llm.with_structured_output(ResearchPlan) # structured_llm will return ResearchPlan schema so we can ignore the plance error below 
        response = structured_llm.invoke(prompt)

        if not isinstance(response, ResearchPlan):
            raise TypeError(
                f"Expected ResearchPlan, got {type(response).__name__}"
            )
        return response

In [ ]:
planner = PlannerAgent()
plan = planner.create_plan(".")
plan


ResearchPlan(query='what are class objects in python', subtasks=[Subtask(id='sync-1', task='Define what a class is in Python and explain its role in object-oriented programming.', mode='web', success_criteria=None), Subtask(id='sync-2', task='Explain how class objects are created and used to instantiate instances in Python.', mode='web', success_criteria=None)])

In [9]:
from core.schemas import EvidenceItem, ResearchPlan
from tools.db_search import RAGSearch
from tools.web_search import web_search



class ResearcherAgent:
    def __init__(self, rag_search: RAGSearch | None = None):
        self.rag_search = rag_search or RAGSearch()

    def run(self, plan: ResearchPlan) -> list[EvidenceItem]:
        findings: list[EvidenceItem] = []

        for subtask in plan.subtasks:
            if subtask.mode == "web":
                findings.extend(web_search(subtask.task))
            elif subtask.mode == "rag":
                findings.extend(self.rag_search.search(subtask.task))
        return findings

ModuleNotFoundError: No module named 'app'

In [6]:
# testing researcher
plan = ResearchPlan(query='what are class objects in python', subtasks=[Subtask(id='sync-1', task='Define what a class is in Python and explain its role in object-oriented programming.', mode='web', success_criteria=None), Subtask(id='sync-2', task='Explain how class objects are created and used to instantiate instances in Python.', mode='web', success_criteria=None)])

In [ ]:
plan.subtasks

'sync-1'